# Direct Python API Smoke Test With Sample `.osz` Files

This notebook exercises the non-CLI interfaces directly.

Flow:
1. Build persistent dataset artifacts from `sample_data/raw/*.osz`.
2. Train the baseline `taiko_transformer` for 1 epoch and resume it to epoch 2.
3. Train the new `taiko_context_transformer` for 1 epoch and resume it to epoch 2.

The cells are intentionally not executed here.


In [1]:
from pathlib import Path

import torch

from src.model.specs import ArchitectureSpec, TrainingSpec
from src.model.train_api import (
    create_training_context,
    load_training_context_from_checkpoint,
    prepare_sample_data_artifacts,
    train_context,
)

repo_root = Path.cwd()
raw_osz_glob = repo_root / "sample_data" / "raw" / "*.osz"
baseline_data_root = repo_root / "sample_data" / "train_api_demo"
context_data_root = repo_root / "sample_data" / "train_api_context_demo"

baseline_training_dir = baseline_data_root / "training"
context_training_dir = context_data_root / "training"

baseline_checkpoint_path = baseline_training_dir / "checkpoints" / "last.ckpt"
context_checkpoint_path = context_training_dir / "checkpoints" / "last.ckpt"

if torch.cuda.is_available():
    best_device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    best_device = "mps"
else:
    best_device = "cpu"

print(f"repo_root               : {repo_root}")
print(f"raw_osz_glob            : {raw_osz_glob}")
print(f"baseline_data_root      : {baseline_data_root}")
print(f"context_data_root       : {context_data_root}")
print(f"baseline_checkpoint     : {baseline_checkpoint_path}")
print(f"context_checkpoint      : {context_checkpoint_path}")
print(f"best_device             : {best_device}")


repo_root               : /Users/accessair/Desktop/Workspaces/taiko-diffusion
raw_osz_glob            : /Users/accessair/Desktop/Workspaces/taiko-diffusion/sample_data/raw/*.osz
baseline_data_root      : /Users/accessair/Desktop/Workspaces/taiko-diffusion/sample_data/train_api_demo
context_data_root       : /Users/accessair/Desktop/Workspaces/taiko-diffusion/sample_data/train_api_context_demo
baseline_checkpoint     : /Users/accessair/Desktop/Workspaces/taiko-diffusion/sample_data/train_api_demo/training/checkpoints/last.ckpt
context_checkpoint      : /Users/accessair/Desktop/Workspaces/taiko-diffusion/sample_data/train_api_context_demo/training/checkpoints/last.ckpt
best_device             : mps


## Baseline Model

### Step 1: Prepare persistent dataset artifacts for the baseline run


In [ ]:
baseline_artifacts = prepare_sample_data_artifacts(
    osz_inputs=[str(raw_osz_glob)],
    data_root=baseline_data_root,
)

print(baseline_artifacts)


Unpacking .osz files: 100%|██████████| 1/1 [00:00<00:00, 7294.44file/s]


### Step 2: Build a direct baseline training context and train to epoch 1


In [ ]:
baseline_architecture_spec = ArchitectureSpec(
    name="taiko_transformer",
    d_model=32,
    nhead=4,
    num_encoder_layers=1,
    num_decoder_layers=1,
    dim_feedforward=64,
    max_len=256,
)

baseline_training_spec = TrainingSpec(
    epochs=1,
    batch_size=4,
    lr=0.001,
    device=best_device,
)

baseline_context = create_training_context(
    data_root=baseline_data_root,
    architecture_spec=baseline_architecture_spec,
    training_spec=baseline_training_spec,
)

print("baseline architecture:", baseline_context.architecture_spec)
print("baseline ignore_index:", baseline_context.dataset.label_ignore_index)
baseline_context = train_context(baseline_context, epochs=1)
assert baseline_checkpoint_path.exists(), "Checkpoint was not written after the baseline API run"


### Step 3: Resume the baseline checkpoint and continue to epoch 2


In [ ]:
baseline_resumed_context = load_training_context_from_checkpoint(
    baseline_checkpoint_path,
    data_root=baseline_data_root,
    device=best_device,
    batch_size=4,
)

print("resumed baseline architecture:", baseline_resumed_context.architecture_spec)
baseline_resumed_context = train_context(baseline_resumed_context, epochs=2)


## Long-Context Model

### Step 4: Prepare persistent dataset artifacts for the long-context run


In [ ]:
context_artifacts = prepare_sample_data_artifacts(
    osz_inputs=[str(raw_osz_glob)],
    data_root=context_data_root,
)

print(context_artifacts)


### Step 5: Build a direct long-context training context and train to epoch 1


In [ ]:
context_architecture_spec = ArchitectureSpec(
    name="taiko_context_transformer",
    d_model=32,
    nhead=4,
    num_encoder_layers=1,
    num_decoder_layers=1,
    dim_feedforward=64,
    max_len=512,
    history_max_tokens=256,
    retrieval_top_k=2,
    retrieval_max_tokens_per_window=32,
    retrieval_exclude_last_n_windows=1,
    use_motif_retrieval=True,
)

context_training_spec = TrainingSpec(
    epochs=1,
    batch_size=4,
    lr=0.001,
    device=best_device,
)

context_context = create_training_context(
    data_root=context_data_root,
    architecture_spec=context_architecture_spec,
    training_spec=context_training_spec,
)

print("context architecture:", context_context.architecture_spec)
print("context ignore_index:", context_context.dataset.label_ignore_index)
context_context = train_context(context_context, epochs=1)
assert context_checkpoint_path.exists(), "Checkpoint was not written after the context API run"


### Step 6: Resume the long-context checkpoint and continue to epoch 2


In [ ]:
context_resumed_context = load_training_context_from_checkpoint(
    context_checkpoint_path,
    data_root=context_data_root,
    device=best_device,
    batch_size=4,
)

print("resumed context architecture:", context_resumed_context.architecture_spec)
context_resumed_context = train_context(context_resumed_context, epochs=2)


## Optional inspection


In [ ]:
print("baseline history keys:", baseline_resumed_context.history.keys())
print("baseline last checkpoint:", baseline_checkpoint_path)
print("baseline vocab json:", baseline_training_dir / "vocab.json")

print("context history keys:", context_resumed_context.history.keys())
print("context architecture:", context_resumed_context.architecture_spec)
print("context label ignore index:", context_resumed_context.dataset.label_ignore_index)
print("context last checkpoint:", context_checkpoint_path)
print("context vocab json:", context_training_dir / "vocab.json")
